# Redrob Candidate Ranking — Sandbox Demo (upload-based)

Reproduces the **ranking step** on a small candidate sample, CPU-only, under 5 minutes, **without cloning** (the repo is private).

**How to run:**
1. On your machine, zip the repo into `repo.zip`.
2. Run **Cell 1** and upload `repo.zip` when prompted.
3. Run the remaining cells top to bottom.

Full 100K reproduction happens in the organizers' Stage-3 environment from the GitHub repo (access granted at Stage 3). This is the small-sample check required by submission_spec Section 10.5.

## 1. Upload the repo zip

In [ ]:
import zipfile, os, glob
from google.colab import files
uploaded = files.upload()  # pick repo.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('repo')
cands = glob.glob('repo/**/rank.py', recursive=True)
assert cands, 'rank.py not found in the uploaded zip'
repo_dir = os.path.dirname(cands[0])
os.chdir(repo_dir)
print('working directory:', os.getcwd())
print('contains:', sorted(os.listdir('.'))[:20])

## 2. Install dependencies (CPU)

In [ ]:
# Install only what the ranking step needs. Do NOT pin numpy/pandas —
# Colab ships numpy 2.x and downgrading mid-session breaks binary compatibility.
# rank.py is version-agnostic, so we use Colab’s existing numpy/pandas.
!pip install -q lightgbm pyarrow

## 3. Build a 100-candidate sample

In [ ]:
import json, pandas as pd
df = pd.read_parquet('artifacts/features_100k.parquet')
sample_ids = list(df.index[:100])
try:
    full = {c['candidate_id']: c for c in json.load(open('sample_candidates.json'))}
except Exception:
    full = {}
with open('sandbox_candidates.jsonl', 'w') as f:
    for cid in sample_ids:
        rec = full.get(cid, {'candidate_id': cid, 'profile': {}, 'career_history': [], 'skills': [], 'redrob_signals': {}})
        f.write(json.dumps(rec) + '\n')
print('wrote', len(sample_ids), 'candidates')

## 4. Run the ranking step (CPU, < 5 min)

In [ ]:
import time
t0 = time.time()
!python rank.py --candidates sandbox_candidates.jsonl --out sandbox_submission.csv
print(f'\nelapsed: {time.time()-t0:.1f}s')

## 5. Show the ranked output

In [ ]:
import pandas as pd
out = pd.read_csv('sandbox_submission.csv')
print('rows:', len(out), '| distinct reasoning:', out.reasoning.nunique())
out.head(15)